In [0]:
print(spark.version)

4.2.0


In [0]:
# 1. Configuración
import math
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.dropdown("ventas", "100000", ["10000", "100000", "500000", "1000000"], "1. Volumen de ventas")
VENTAS = int(dbutils.widgets.get("ventas"))

CATALOGO = "workspace"
ESQUEMA  = "ceramicas_vale"
VOLUMEN  = "datos"
RUTA     = "/Volumes/workspace/bigdata/ef/ARCHIVOS"

ARCHIVO_CAJA = {10000: "operacionVenta_10000.txt", 100000: "operacionVenta.txt",
                500000: "operacionVenta_500000.txt", 1000000: "operacionVenta_1000000.txt"}[VENTAS]

def tabla(nombre):
    return f"{CATALOGO}.{ESQUEMA}.{nombre}"

print(f"Volumen: {VENTAS:,} ventas | archivo de caja: {ARCHIVO_CAJA}")

Volumen: 10,000 ventas | archivo de caja: operacionVenta_10000.txt


In [0]:
# 2. Esquema, volumen y revisión de archivos
spark.sql(f"create schema if not exists {CATALOGO}.{ESQUEMA}")
spark.sql(f"create volume if not exists {CATALOGO}.{ESQUEMA}.{VOLUMEN}")
spark.sql(f"use catalog {CATALOGO}")
spark.sql(f"use schema {ESQUEMA}")

necesarios = ["proveedores.csv", "productos.csv", "clientes.csv",
              f"ventas_{VENTAS}.csv", f"detalle_ventas_{VENTAS}.csv",
              "metodopago.txt", ARCHIVO_CAJA]
subidos = [f.name for f in dbutils.fs.ls(RUTA)]
faltan = [a for a in necesarios if a not in subidos]
print("FALTAN:", faltan) if faltan else print("Todos los archivos están en", RUTA)

Todos los archivos están en /Volumes/workspace/bigdata/ef/ARCHIVOS


In [0]:
# 3. Funciones de apoyo
bitacora = []
INICIO_ETL = datetime.now()

ID = 1
if spark.catalog.tableExists(tabla("etl_ejecucion")):
    ID = (spark.table(tabla("etl_ejecucion")).agg(F.max("id_ejecucion")).first()[0] or 0) + 1

def guardar(df, nombre):
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabla(nombre))
    return spark.table(tabla(nombre)).count()

def registrar_errores(df):
    (df.select(F.lit(ID).alias("id_ejecucion"), "origen", F.col("clave").cast("string").alias("clave"), "motivo")
       .write.mode("append").saveAsTable(tabla("etl_errores")))

def paso(proceso, inicio, filas, estado="EXITOSO", mensaje=None):
    fin = datetime.now()
    ms = int((fin - inicio).total_seconds() * 1000)
    bitacora.append((ID, proceso, inicio, fin, ms, int(filas), estado, mensaje))
    print(f"{proceso:<28} {ms:>8,} ms {int(filas):>12,} filas  {estado}")

def texto(columna):
    return F.trim(F.regexp_replace(F.col(columna), "[\\x00\\r]", ""))

def vacio_a_nulo(col):
    return F.when(col == "", None).otherwise(col)

print("Ejecución N.°", ID)

Ejecución N.° 1


In [0]:
# Paso 1. Extracción (raw_*)
inicio = datetime.now()
lector = spark.read.option("sep", "|").option("header", "false").option("encoding", "UTF-8")

origenes = {
    "raw_proveedores":    ("proveedores.csv", "id_proveedor int, ruc string, razon_social string, telefono string, ciudad string"),
    "raw_productos":      ("productos.csv", "codigo_interno string, descripcion string, unidad_medida string, marca string, "
                                            "precio decimal(10,2), categoria string, m2_por_caja decimal(6,2), "
                                            "costo_unitario decimal(10,2), stock_actual int, stock_minimo int, "
                                            "estado string, id_proveedor int"),
    "raw_clientes":       ("clientes.csv", "id_cliente int, tipo_documento string, nro_documento string, nombre string, distrito string"),
    "raw_ventas":         (f"ventas_{VENTAS}.csv", "id_venta int, tipo_comprobante string, serie_numero string, fecha timestamp, "
                                                   "id_cliente int, estado string, total decimal(12,2)"),
    "raw_detalle_ventas": (f"detalle_ventas_{VENTAS}.csv", "id_venta int, item int, codigo_interno string, cantidad decimal(10,2), "
                                                           "precio_unitario decimal(10,2), descuento_pct decimal(4,2)"),
}
filas = 0
for nombre, (archivo, esquema) in origenes.items():
    df = lector.schema(esquema).option("timestampFormat", "yyyy-MM-dd HH:mm:ss").csv(f"{RUTA}/{archivo}")
    filas += guardar(df, nombre)
paso("1. Extracción (raw_*)", inicio, filas)

1. Extracción (raw_*)           9,851 ms       37,436 filas  EXITOSO


In [0]:
# Paso 2. dim_proveedor
inicio = datetime.now()
dim_proveedor = (spark.table(tabla("raw_proveedores"))
    .withColumnRenamed("id_proveedor", "pk_proveedor")
    .select("pk_proveedor",
            texto("ruc").alias("ruc"),
            F.upper(texto("razon_social")).alias("razon_social"),
            vacio_a_nulo(F.upper(texto("ciudad"))).alias("ciudad"))
    .na.fill({"ciudad": "NO ESPECIFICADO"})
    .withColumn("id_proveedor", F.row_number().over(Window.orderBy("pk_proveedor")))
    .select("id_proveedor", "pk_proveedor", "ruc", "razon_social", "ciudad"))
paso("2. dim_proveedor", inicio, guardar(dim_proveedor, "dim_proveedor"))
display(spark.table(tabla("dim_proveedor")))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


2. dim_proveedor                1,858 ms           50 filas  EXITOSO


id_proveedor,pk_proveedor,ruc,razon_social,ciudad
1,1,20682814345,DISTRIBUIDORA APACHE S.A.C.,LIMA
2,2,20681584390,DISTRIBUIDORA BELLAPLAST S.A.C.,LIMA
3,3,20229360405,DISTRIBUIDORA CASTLE S.A.C.,LIMA
4,4,20830779084,DISTRIBUIDORA CELIMA S.A.C.,LIMA
5,5,20206734731,DISTRIBUIDORA CHEMA S.A.C.,LIMA
6,6,20218362394,DISTRIBUIDORA CHEMICAL S.A.C.,LIMA
7,7,20950717323,DISTRIBUIDORA DACQUA S.A.C.,CALLAO
8,8,20486229131,DISTRIBUIDORA DIROFASA S.A.C.,LIMA
9,9,20132867324,DISTRIBUIDORA ECKOGRIF S.A.C.,LIMA
10,10,20580021743,DISTRIBUIDORA EL DORADO S.A.C.,LIMA


In [0]:
# Paso 3. dim_producto
def categoria(descripcion, unidad):
    d, u = F.col(descripcion), F.col(unidad)
    def tiene(*palabras):
        cond = F.lit(False)
        for p in palabras:
            cond = cond | d.contains(p)
        return cond
    return (F.when(u.isin("MTK", "BX") & tiene("PORCELANATO"), "PORCELANATOS")
             .when(u.isin("MTK", "BX"), "CERAMICOS")
             .when(tiene("INODORO", "INNODORO", "LAVATORIO", "OVALIN", "LAVADERO", "URINARIO", "ONE PIECE", "TANQUE"), "SANITARIOS")
             .when(tiene("DUCHA", "MEZCLADORA", "PICO DE GANZO", "GRIFO", "LLAVE", "MONOCOMANDO", "GRIFERIA"), "GRIFERIA")
             .when(tiene("PEGAMENTO", "FRAGUA", "SILICONA", "CEMENTO", "TEROKAL"), "PEGAMENTOS Y FRAGUAS")
             .when(tiene("PERFIL", "CUÑA", "NIVELADOR", "CANTONERA") | (u == "MTR"), "PERFILES Y NIVELACION")
             .when(tiene("DECORADO", "DECO ", "LISTELO", "PEPELMA", "CENEFA", "MOSAICO"), "DECORADOS")
             .otherwise("ACCESORIOS Y OTROS"))

inicio = datetime.now()
productos = spark.table(tabla("raw_productos"))

registrar_errores(productos.filter(F.col("precio") <= 1)
                  .select(F.lit("PRODUCTOS").alias("origen"), F.col("codigo_interno").alias("clave"),
                          F.lit("Precio <= 1 (valor de prueba)").alias("motivo")))

dim_producto = (productos
    .filter(F.col("precio") > 1)
    .withColumnRenamed("codigo_interno", "pk_producto")
    .select(texto("pk_producto").alias("pk_producto"),
            F.trim(F.regexp_replace(F.upper(texto("descripcion")), "\\s+", " ")).alias("descripcion"),
            F.upper(texto("marca")).alias("marca"),
            texto("unidad_medida").alias("unidad_medida"),
            "m2_por_caja", "precio", "costo_unitario", "stock_actual", "stock_minimo",
            F.when(F.col("stock_actual") <= F.col("stock_minimo"), "CRITICO").otherwise("NORMAL").alias("estado_stock"))
    .withColumn("categoria", categoria("descripcion", "unidad_medida"))
    .withColumn("id_producto", F.row_number().over(Window.orderBy("pk_producto")))
    .select("id_producto", "pk_producto", "descripcion", "categoria", "marca", "unidad_medida", "m2_por_caja",
            "precio", "costo_unitario", "stock_actual", "stock_minimo", "estado_stock"))
paso("3. dim_producto", inicio, guardar(dim_producto, "dim_producto"))
display(spark.table(tabla("dim_producto")))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


3. dim_producto                 3,366 ms        2,028 filas  EXITOSO


id_producto,pk_producto,descripcion,categoria,marca,unidad_medida,m2_por_caja,precio,costo_unitario,stock_actual,stock_minimo,estado_stock
1,01052,SIGNO AZUL 27X45 (2.07 X CAJ) X 17 PZA,CERAMICOS,GENÉRICO,MTK,2.07,28.00,19.32,161,23,NORMAL
2,01054,LLAVE DE DUCHA TREBOL LINEA ECO,GRIFERIA,TREBOL,NIU,null,55.00,37.40,58,29,NORMAL
3,01057,RUEDA DE CORTADORA MARCA RUBÍ,ACCESORIOS Y OTROS,GENÉRICO,NIU,null,80.00,53.60,180,10,NORMAL
4,01060,RUEDA DE CORTADORA MARCA JAMTEXS,ACCESORIOS Y OTROS,GENÉRICO,NIU,null,45.00,27.90,60,36,NORMAL
5,01061,ROBLE CARAMELO 45X45 (2.03X10PZAS) - COMERCIAL,CERAMICOS,GENÉRICO,MTK,2.03,26.00,19.24,181,35,NORMAL
6,01063,DALLAS BEIGE 45X45 COMERCIAL (2.03X10PZAS) - COMERCIAL,CERAMICOS,GENÉRICO,MTK,2.03,24.00,14.64,140,11,NORMAL
7,01077,PARED AMAPOLA HUESO 30X60 (1.81X10PZAS),CERAMICOS,GENÉRICO,MTK,1.81,35.00,26.25,194,32,NORMAL
8,01078,RUBIC LUCES PLATA 27X45 (2.07 X CAJ) X 17 PZAS,CERAMICOS,GENÉRICO,MTK,2.07,29.00,20.59,62,40,NORMAL
9,01079,SIGNO GRIS 27X45 (2.07X CAJ) X 17 PZAS,CERAMICOS,GENÉRICO,MTK,2.07,27.00,18.63,84,13,NORMAL
10,01081,DUNAS BLANCO BRILLANTE 25X40 (2.03 X CAJ) X 20 PZA,CERAMICOS,GENÉRICO,MTK,2.03,26.00,19.50,67,34,NORMAL


In [0]:
# Paso 4. dim_cliente
inicio = datetime.now()
clientes = spark.table(tabla("raw_clientes"))
ventas   = spark.table(tabla("raw_ventas"))

varios  = clientes.filter(texto("nro_documento") == "00000000").select("id_cliente")
compras = (ventas.filter(texto("estado") == "EMITIDO")
                 .join(varios, "id_cliente", "left_anti")
                 .groupBy("id_cliente").agg(F.sum("total").alias("total")))
n_vip = math.ceil(compras.count() * 0.10)
vip = (compras.orderBy(F.desc("total")).limit(n_vip)
              .select(F.col("id_cliente").alias("pk_cliente"), F.lit("VIP").alias("vip")))

dim_cliente = (clientes
    .withColumnRenamed("id_cliente", "pk_cliente")
    .select("pk_cliente",
            texto("tipo_documento").alias("tipo_documento"),
            texto("nro_documento").alias("nro_documento"),
            F.upper(F.regexp_replace(texto("nombre"), " {2,}", " ")).alias("nombre"),
            vacio_a_nulo(F.upper(texto("distrito"))).alias("distrito"))
    .na.fill({"distrito": "NO ESPECIFICADO"})
    .join(vip, "pk_cliente", "left")
    .withColumn("tipo_cliente", F.coalesce("vip", F.lit("REGULAR")))
    .withColumn("id_cliente", F.row_number().over(Window.orderBy("pk_cliente")))
    .select("id_cliente", "pk_cliente", "tipo_documento", "nro_documento", "nombre", "distrito", "tipo_cliente"))
paso("4. dim_cliente", inicio, guardar(dim_cliente, "dim_cliente"))
display(spark.table(tabla("dim_cliente")).groupBy("tipo_cliente").count())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


4. dim_cliente                  2,795 ms          400 filas  EXITOSO


tipo_cliente,count
REGULAR,360
VIP,40


In [0]:
# Paso 5. dim_tiempo
inicio = datetime.now()
MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto",
         "Septiembre", "Octubre", "Noviembre", "Diciembre"]
DIAS  = ["Domingo", "Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado"]

dim_tiempo = (ventas
    .agg(F.min(F.to_date("fecha")).alias("ini"), F.max(F.to_date("fecha")).alias("fin"))
    .select(F.explode(F.sequence("ini", "fin")).alias("fecha"))
    .select("fecha",
            F.year("fecha").alias("anio"),
            F.quarter("fecha").alias("trimestre"),
            F.month("fecha").alias("mes"),
            F.element_at(F.array(*[F.lit(m) for m in MESES]), F.month("fecha")).alias("nombreMes"),
            F.weekofyear("fecha").alias("semana"),
            F.dayofmonth("fecha").alias("diaMes"),
            F.element_at(F.array(*[F.lit(d) for d in DIAS]), F.dayofweek("fecha")).alias("diaSemana")))
paso("5. dim_tiempo", inicio, guardar(dim_tiempo, "dim_tiempo"))

5. dim_tiempo                   1,831 ms        1,022 filas  EXITOSO


In [0]:
# Paso 6. dim_metodoPago (metodopago.txt)
inicio = datetime.now()
stg_metodo = (spark.read.option("sep", ",").option("header", "false")
                   .schema("pk_metodoP string, nombreMetodo string")
                   .csv(f"{RUTA}/metodopago.txt"))
guardar(stg_metodo, "stg_metodoPago")
display(spark.table(tabla("stg_metodoPago")))

no_registrado = spark.createDataFrame([(0, "NO REGISTRADO")], "pk_metodoP int, nombreMetodo string")
dim_metodo = (spark.table(tabla("stg_metodoPago"))
    .select(F.expr("try_cast(trim(pk_metodoP) as int)").alias("pk_metodoP"),
            F.upper(texto("nombreMetodo")).alias("nombreMetodo"))
    .distinct()
    .unionByName(no_registrado)
    .withColumn("id_metodoP", F.row_number().over(Window.orderBy("pk_metodoP")))
    .select("id_metodoP", "pk_metodoP", "nombreMetodo"))
paso("6. dim_metodoPago (txt)", inicio, guardar(dim_metodo, "dim_metodoPago"))
display(spark.table(tabla("dim_metodoPago")))

pk_metodoP,nombreMetodo
1,Efectivo
2,yape/plin
3,Tarjeta
4,Transferencia
5,Deposito en cuenta
3,Tarjeta


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


6. dim_metodoPago (txt)         4,414 ms            6 filas  EXITOSO


id_metodoP,pk_metodoP,nombreMetodo
1,0,NO REGISTRADO
2,1,EFECTIVO
3,2,YAPE/PLIN
4,3,TARJETA
5,4,TRANSFERENCIA
6,5,DEPOSITO EN CUENTA


In [0]:
# Paso 7. stg_operacion (archivo de caja)
inicio = datetime.now()
stg_operacion = (spark.read.option("sep", "|").option("header", "true")
                      .schema("comprobante string, metodoPago string")
                      .csv(f"{RUTA}/{ARCHIVO_CAJA}")
                      .select(texto("comprobante").alias("comprobante"), texto("metodoPago").alias("metodoPago")))
filas = guardar(stg_operacion, "stg_operacion")
stg_operacion = spark.table(tabla("stg_operacion"))

registrar_errores(stg_operacion.groupBy("comprobante").count().filter(F.col("count") > 1)
                  .select(F.lit("operacionVenta.txt").alias("origen"), F.col("comprobante").alias("clave"),
                          F.lit("Comprobante duplicado en archivo de caja").alias("motivo")))

registrar_errores(stg_operacion
                  .withColumn("pk_metodoP", F.expr("try_cast(metodoPago as int)"))
                  .join(spark.table(tabla("dim_metodoPago")).select("pk_metodoP"), "pk_metodoP", "left_anti")
                  .select(F.lit("operacionVenta.txt").alias("origen"), F.col("comprobante").alias("clave"),
                          F.concat(F.lit("Método de pago inexistente: "), F.col("metodoPago")).alias("motivo")))
paso("7. stg_operacion (txt)", inicio, filas)

7. stg_operacion (txt)          4,687 ms        9,959 filas  EXITOSO


In [0]:
# Paso 8. fac_ventas
inicio = datetime.now()
V = (spark.table(tabla("raw_ventas"))
          .withColumnRenamed("id_cliente", "pk_cliente")
          .withColumn("estado", texto("estado"))
          .withColumn("serie_numero", texto("serie_numero")))
D = spark.table(tabla("raw_detalle_ventas")).withColumn("codigo_interno", texto("codigo_interno"))
P = (spark.table(tabla("raw_productos"))
          .select(texto("codigo_interno").alias("codigo_interno"), texto("unidad_medida").alias("unidad_medida"),
                  "m2_por_caja", "costo_unitario", F.col("id_proveedor").alias("pk_proveedor")))
VD = V.join(D, "id_venta")
emitido = F.col("estado") == "EMITIDO"
validas = (F.col("precio_unitario") > 0) & (F.col("cantidad") > 0)

registrar_errores(VD.filter(emitido & ~validas)
                  .select(F.lit("DETALLE_VENTAS").alias("origen"),
                          F.concat("serie_numero", F.lit(" / ítem "), F.col("item").cast("string")).alias("clave"),
                          F.lit("Precio o cantidad <= 0").alias("motivo")))

registrar_errores(V.filter(emitido)
                  .join(stg_operacion.select(F.col("comprobante").alias("serie_numero")), "serie_numero", "left_anti")
                  .select(F.lit("VENTAS").alias("origen"), F.col("serie_numero").alias("clave"),
                          F.lit("Comprobante sin registro de pago (se carga como NO REGISTRADO)").alias("motivo")))

dim_metodo     = spark.table(tabla("dim_metodoPago"))
id_no_registro = dim_metodo.filter(F.col("pk_metodoP") == 0).first()["id_metodoP"]
metodo_por_comprobante = (stg_operacion.groupBy(F.col("comprobante").alias("serie_numero"))
                          .agg(F.min(F.expr("try_cast(metodoPago as int)")).alias("pk_metodoP")))

cant, m2, um = F.col("cantidad"), F.col("m2_por_caja"), F.col("unidad_medida")
precio, desc = F.col("precio_unitario"), F.col("descuento_pct")
divisor = F.when(m2.isNull() | (m2 == 0), F.lit(1)).otherwise(m2)

fac_ventas = (VD.filter(emitido & validas)
    .join(P, "codigo_interno")
    .join(spark.table(tabla("dim_producto")).select(F.col("pk_producto").alias("codigo_interno"), "id_producto"), "codigo_interno")
    .join(spark.table(tabla("dim_proveedor")).select("pk_proveedor", "id_proveedor"), "pk_proveedor")
    .join(spark.table(tabla("dim_cliente")).select("pk_cliente", "id_cliente"), "pk_cliente")
    .join(metodo_por_comprobante, "serie_numero", "left")
    .join(dim_metodo.select("pk_metodoP", "id_metodoP"), "pk_metodoP", "left")
    .withColumn("id_metodoP", F.coalesce("id_metodoP", F.lit(id_no_registro)))
    .groupBy("id_cliente", "id_producto", "id_proveedor", "id_metodoP", F.to_date("fecha").alias("fecha"))
    .agg(F.sum(F.when(um == "MTK", cant / divisor).otherwise(cant)).alias("cantidad"),
         F.sum(cant * precio * (1 - desc)).alias("monto_vendido"),
         F.sum(F.when(um == "MTK", cant).when(um == "BX", cant * F.coalesce(m2, F.lit(0))).otherwise(F.lit(0))).alias("metraje_vendido"),
         F.sum(cant * F.col("costo_unitario")).alias("costo_inventario"),
         F.sum(cant * precio * desc).alias("descuento")))
paso("8. fac_ventas", inicio, guardar(fac_ventas, "fac_ventas"))
display(spark.table(tabla("fac_ventas")).limit(100))

8. fac_ventas                   6,470 ms       24,387 filas  EXITOSO


id_cliente,id_producto,id_proveedor,id_metodoP,fecha,cantidad,monto_vendido,metraje_vendido,costo_inventario,descuento
176,1014,48,4,2025-08-28,20.000000000000000,690.200000,40.6000,496.9440,0.000000
265,587,41,2,2026-01-11,3.000000000000000,450.000000,0.0000,319.5000,0.000000
1,792,48,3,2025-10-27,4.000000000000000,300.000000,0.0000,198.0000,0.000000
1,903,49,2,2026-01-19,20.000000000000000,810.000000,30.0000,494.1000,0.000000
112,27,16,3,2025-04-04,2.000000000000000,560.000000,0.0000,352.8000,0.000000
26,905,49,5,2025-09-20,5.000000000000000,45.000000,0.0000,32.8500,0.000000
1,863,50,4,2026-03-14,11.000000000000000,154.000000,0.0000,97.0200,0.000000
246,295,47,2,2026-06-21,12.000000000000000,360.000000,0.0000,262.8000,0.000000
132,783,49,5,2026-10-08,2.000000000000000,240.000000,0.0000,151.2000,0.000000
309,869,50,2,2026-09-12,17.000000000000000,529.380000,25.5000,328.1850,0.000000


In [0]:
# Paso 9. Reconciliación
inicio = datetime.now()
origen = (VD.filter(emitido & validas)
            .join(spark.table(tabla("raw_productos")).filter(F.col("precio") > 1)
                       .select(texto("codigo_interno").alias("codigo_interno")), "codigo_interno")
            .agg(F.sum(cant * precio * (1 - desc))).first()[0])
destino = spark.table(tabla("fac_ventas")).agg(F.sum("monto_vendido")).first()[0]
origen, destino = float(origen or 0), float(destino or 0)
dif = destino - origen
estado_rec = "OK" if abs(dif) <= abs(origen) * 0.000001 + 0.01 else "DIFERENCIA"
mensaje = f"Origen S/ {origen:,.2f} | Databricks S/ {destino:,.2f} | Diferencia S/ {dif:,.4f}"
filas_hechos = spark.table(tabla("fac_ventas")).count()
paso("9. Reconciliación", inicio, filas_hechos, estado_rec, mensaje)
print(mensaje)

9. Reconciliación               1,285 ms       24,387 filas  OK
Origen S/ 11,697,928.53 | Databricks S/ 11,697,928.53 | Diferencia S/ 0.0000


In [0]:
# Registro de la ejecución
fin_etl = datetime.now()
pasos_ok = sum(1 for b in bitacora if b[6] == "EXITOSO" and not b[1].startswith("9."))
estado_final = "EXITOSO" if pasos_ok == 8 and estado_rec == "OK" else "ERROR"

(spark.createDataFrame(bitacora, "id_ejecucion int, proceso string, hora_inicio timestamp, hora_fin timestamp, "
                                 "duracion_ms long, filas_procesadas long, estado string, mensaje string")
      .write.mode("append").saveAsTable(tabla("etl_bitacora")))
(spark.createDataFrame([(ID, f"{VENTAS} ventas", VENTAS, INICIO_ETL, fin_etl,
                         int((fin_etl - INICIO_ETL).total_seconds() * 1000), filas_hechos, estado_final,
                         None if estado_final == "EXITOSO" else f"{pasos_ok} de 8 pasos exitosos; reconciliación {estado_rec}")],
                       "id_ejecucion int, escenario string, ventas int, fecha_inicio timestamp, fecha_fin timestamp, "
                       "duracion_ms long, filas_hechos long, estado string, mensaje string")
      .write.mode("append").saveAsTable(tabla("etl_ejecucion")))

print(f"Ejecución {ID}: {estado_final} en {(fin_etl - INICIO_ETL).total_seconds():.2f} s")
display(spark.table(tabla("etl_bitacora")).filter(F.col("id_ejecucion") == ID)
             .orderBy("hora_inicio").select("proceso", "duracion_ms", "filas_procesadas", "estado", "mensaje"))
display(spark.table(tabla("etl_errores")).filter(F.col("id_ejecucion") == ID)
             .groupBy("origen", "motivo").count().orderBy("origen"))

Ejecución 1: EXITOSO en 45.73 s


proceso,duracion_ms,filas_procesadas,estado,mensaje
1. Extracción (raw_*),9851,37436,EXITOSO,null
2. dim_proveedor,1858,50,EXITOSO,null
3. dim_producto,3366,2028,EXITOSO,null
4. dim_cliente,2795,400,EXITOSO,null
5. dim_tiempo,1831,1022,EXITOSO,null
6. dim_metodoPago (txt),4414,6,EXITOSO,null
7. stg_operacion (txt),4687,9959,EXITOSO,null
8. fac_ventas,6470,24387,EXITOSO,null
9. Reconciliación,1285,24387,OK,"Origen S/ 11,697,928.53 | Databricks S/ 11,697,928.53 | Diferencia S/ 0.0000"


origen,motivo,count
DETALLE_VENTAS,Precio o cantidad <= 0,5
PRODUCTOS,Precio <= 1 (valor de prueba),17
VENTAS,Comprobante sin registro de pago (se carga como NO REGISTRADO),47
operacionVenta.txt,Comprobante duplicado en archivo de caja,6
operacionVenta.txt,Método de pago inexistente: 9,10


In [0]:
# Calidad de datos: antes vs después
def contar(df, condicion):
    return df.filter(condicion).count()

raw_cli, raw_prod = spark.table(tabla("raw_clientes")), spark.table(tabla("raw_productos"))
dim_cli, dim_prod = spark.table(tabla("dim_cliente")), spark.table(tabla("dim_producto"))
nombre_sucio = lambda c: (F.col(c) != F.upper(F.col(c))) | F.col(c).contains("  ")

calidad = [
    ("Clientes con nombre en minúsculas o espacios dobles", contar(raw_cli, nombre_sucio("nombre")), contar(dim_cli, nombre_sucio("nombre"))),
    ("Clientes sin distrito (después: NO ESPECIFICADO)",
        contar(raw_cli, F.col("distrito").isNull() | (texto("distrito") == "")), contar(dim_cli, F.col("distrito") == "NO ESPECIFICADO")),
    ("Productos sin categoría", contar(raw_prod, F.col("categoria").isNull() | (texto("categoria") == "")), contar(dim_prod, F.col("categoria").isNull())),
    ("Descripciones con minúsculas", contar(raw_prod, F.col("descripcion") != F.upper(F.col("descripcion"))),
        contar(dim_prod, F.col("descripcion") != F.upper(F.col("descripcion")))),
    ("Productos con precio de prueba (<= 1)", contar(raw_prod, F.col("precio") <= 1), contar(dim_prod, F.col("precio") <= 1)),
]
display(spark.createDataFrame(calidad, "control string, antes long, despues long"))

fac = spark.table(tabla("fac_ventas"))
display(fac.select("cantidad", "monto_vendido", "metraje_vendido", "costo_inventario", "descuento").describe())
display(fac.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in fac.columns]))

control,antes,despues
Clientes con nombre en minúsculas o espacios dobles,23,0
Clientes sin distrito (después: NO ESPECIFICADO),17,17
Productos sin categoría,1957,0
Descripciones con minúsculas,290,0
Productos con precio de prueba (<= 1),17,0


summary,cantidad,monto_vendido,metraje_vendido,costo_inventario,descuento
count,24387,24387,24387,24387,24387
mean,10.2307786935662443105,479.6788672235,11.97415344,325.31963699,4.7466871620
stddev,8.246074938226378,501.3360779458328,16.565504094847874,338.10469281631606,18.35802157166715
min,1.000000000000000,2.000000,0.0000,1.2200,0.000000
max,59.000000000000000,8700.000000,106.2000,5220.0000,315.900000


id_cliente,id_producto,id_proveedor,id_metodoP,fecha,cantidad,monto_vendido,metraje_vendido,costo_inventario,descuento
0,0,0,0,0,0,0,0,0,0


In [0]:
%sql
-- KPI: ventas por año y mes
select T.anio, T.mes, T.nombreMes, sum(F.monto_vendido) monto, sum(F.cantidad) cantidad
from fac_ventas F join dim_tiempo T on T.fecha = F.fecha
group by T.anio, T.mes, T.nombreMes order by T.anio, T.mes

anio,mes,nombreMes,monto,cantidad
2024,1,Enero,132715.098900,2980.000000000000000
2024,2,Febrero,179788.412200,3912.000000000000000
2024,3,Marzo,224173.132220,4688.000000000000000
2024,4,Abril,286394.589850,6018.000000000000000
2024,5,Mayo,230262.371750,4620.000000000000000
2024,6,Junio,286307.726260,5911.000000000000000
2024,7,Julio,305710.671900,6561.000000000000000
2024,8,Agosto,356205.543150,7439.000000000000000
2024,9,Septiembre,291490.374020,6376.000000000000000
2024,10,Octubre,335809.196400,7405.000000000000000


In [0]:
%sql
-- KPI: ventas, metraje y margen por categoría
select P.categoria, sum(F.monto_vendido) monto, sum(F.metraje_vendido) m2,
       sum(F.monto_vendido - F.costo_inventario) margen,
       round(100 * sum(F.monto_vendido - F.costo_inventario) / sum(F.monto_vendido), 2) margen_pct
from fac_ventas F join dim_producto P on P.id_producto = F.id_producto
group by P.categoria order by monto desc

categoria,monto,m2,margen,margen_pct
CERAMICOS,6042019.863500,236199.3000,1923787.732000,31.84
PORCELANATOS,2354260.344480,55814.3800,779302.738680,33.10
SANITARIOS,1269919.100000,0.0000,409885.680000,32.28
ACCESORIOS Y OTROS,1153148.100000,0.0000,366475.960000,31.78
GRIFERIA,378037.350000,0.0000,124121.890000,32.83
PEGAMENTOS Y FRAGUAS,218237.840000,0.0000,70633.660000,32.37
DECORADOS,166353.870000,0.0000,51948.280000,31.23
PERFILES Y NIVELACION,115952.067000,0.0000,38202.607000,32.95


In [0]:
%sql
-- KPI: participación por método de pago
select M.nombreMetodo, sum(F.monto_vendido) monto,
       round(100 * sum(F.monto_vendido) / (select sum(monto_vendido) from fac_ventas), 2) participacion_pct
from fac_ventas F join dim_metodoPago M on M.id_metodoP = F.id_metodoP
group by M.nombreMetodo order by monto desc

nombreMetodo,monto,participacion_pct
EFECTIVO,3899748.937090,33.34
YAPE/PLIN,3592048.414160,30.71
TARJETA,1767894.534990,15.11
TRANSFERENCIA,1762993.007340,15.07
DEPOSITO EN CUENTA,612180.389400,5.23
NO REGISTRADO,63063.252000,0.54


In [0]:
%sql
-- KPI: top 10 proveedores
select V.razon_social, sum(F.monto_vendido) monto, sum(F.costo_inventario) costo
from fac_ventas F join dim_proveedor V on V.id_proveedor = F.id_proveedor
group by V.razon_social order by monto desc limit 10

razon_social,monto,costo
IMPORT ASIA CERAMICS S.A.C.,2532154.518330,1683923.8920
IMPORTADORA ANDINA DE ACABADOS S.A.C.,2346516.795750,1587969.0648
COMERCIAL LOS OLIVOS S.R.L.,2138572.625900,1483871.3838
DISTRIBUIDORA DEL NORTE E.I.R.L.,1770701.513500,1219452.9755
DISTRIBUIDORA CELIMA S.A.C.,1485890.926500,997295.7868
DISTRIBUIDORA FRATELLI S.A.C.,204585.050000,137861.5600
DISTRIBUIDORA TREBOL S.A.C.,200806.750000,136291.6500
DISTRIBUIDORA VAINSA S.A.C.,118826.000000,81670.2200
DISTRIBUIDORA SAN LORENZO S.A.C.,109391.820000,73734.4864
DISTRIBUIDORA FRENCIA S.A.C.,64159.500000,43272.2000


In [0]:
%sql
-- KPI: clientes VIP vs regulares
select C.tipo_cliente, count(distinct C.id_cliente) clientes, sum(F.monto_vendido) monto
from fac_ventas F join dim_cliente C on C.id_cliente = F.id_cliente
group by C.tipo_cliente

tipo_cliente,clientes,monto
VIP,40,2344042.476790
REGULAR,360,9353886.058190


In [0]:
%sql
-- KPI: productos en stock crítico
select categoria, count(*) productos_criticos
from dim_producto where estado_stock = 'CRITICO'
group by categoria order by productos_criticos desc

categoria,productos_criticos
ACCESORIOS Y OTROS,67
SANITARIOS,51
CERAMICOS,42
PEGAMENTOS Y FRAGUAS,32
PORCELANATOS,18
PERFILES Y NIVELACION,15
GRIFERIA,13
DECORADOS,9


In [0]:
%sql
-- Experimento BD-1: resumen por volumen
select ventas, count(*) ejecuciones, max(filas_hechos) filas_hechos,
       round(avg(duracion_ms) / 1000, 2) promedio_s,
       round(stddev(duracion_ms) / 1000, 2) desviacion_s,
       round(min(duracion_ms) / 1000, 2) minimo_s,
       round(max(duracion_ms) / 1000, 2) maximo_s,
       round(max(filas_hechos) / (avg(duracion_ms) / 1000)) filas_por_segundo
from etl_ejecucion
where estado = 'EXITOSO'
group by ventas order by ventas

ventas,ejecuciones,filas_hechos,promedio_s,desviacion_s,minimo_s,maximo_s,filas_por_segundo
10000,1,24387,45.73,null,45.73,45.73,533.0


In [0]:
%sql
-- Experimento BD-1: tiempo promedio por paso y volumen
select E.ventas, B.proceso, round(avg(B.duracion_ms)) ms_promedio
from etl_bitacora B join etl_ejecucion E on E.id_ejecucion = B.id_ejecucion
where E.estado = 'EXITOSO'
group by E.ventas, B.proceso order by E.ventas, B.proceso

ventas,proceso,ms_promedio
10000,1. Extracción (raw_*),9851.0
10000,2. dim_proveedor,1858.0
10000,3. dim_producto,3366.0
10000,4. dim_cliente,2795.0
10000,5. dim_tiempo,1831.0
10000,6. dim_metodoPago (txt),4414.0
10000,7. stg_operacion (txt),4687.0
10000,8. fac_ventas,6470.0
10000,9. Reconciliación,1285.0


In [0]:
%sql
-- Experimento BD-1: detalle de cada corrida
select E.id_ejecucion, E.ventas, E.fecha_inicio, round(E.duracion_ms / 1000, 2) segundos,
       E.filas_hechos, E.estado, B.mensaje reconciliacion
from etl_ejecucion E
join etl_bitacora B on B.id_ejecucion = E.id_ejecucion and B.proceso like '9.%'
order by E.id_ejecucion

id_ejecucion,ventas,fecha_inicio,segundos,filas_hechos,estado,reconciliacion
1,10000,2026-11-27T01:51:29.963Z,45.73,24387,EXITOSO,"Origen S/ 11,697,928.53 | Databricks S/ 11,697,928.53 | Diferencia S/ 0.0000"


In [0]:
%sql
use catalog workspace;
use schema ceramicas_vale;

delete from etl_errores
where id_ejecucion in (select id_ejecucion from etl_ejecucion where ventas = 500000);

delete from etl_bitacora
where id_ejecucion in (select id_ejecucion from etl_ejecucion where ventas = 500000);

delete from etl_ejecucion
where ventas = 500000;

num_affected_rows
0
